# This notebook uses a ResNet50 model

## Imports

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, random_split
from torch.optim.lr_scheduler import ReduceLROnPlateau



from torchvision import transforms, models

from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import copy

from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,classification_report
import numpy as np
from sklearn.model_selection import StratifiedKFold


from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

import pandas as pd


### Working directory setup

In [2]:
# Mappa elérési útja
results_dir = '/kaggle/working/results'
results_dir = '/kaggle/working/results/test_data'

# Ellenőrizd, hogy a mappa létezik-e, és hozd létre, ha nem
if not os.path.exists(results_dir):
    os.makedirs(results_dir)
    print(f"Directory '{results_dir}' created.")

Directory '/kaggle/working/results/test_data' created.


## Dataset class definition, inherited from PyTorch dataset class
A megadott Python osztály, ```MaskedImageDataset``` a PyTorch beépített Dataset osztályából származik, és képes képadatok és hozzájuk tartozó maszkok kezelésére. Az osztály ``__getitem__`` felülírt függvényének  kimenete az adathalmazban levő bináris maszkok felhasználásával adja vissza a maszkolt képeket. 

In [3]:
class MaskedImageDataset(Dataset):
    def __init__(self, image_paths=None, mask_paths=None, labels=None, class_names=None, transform=None):
        """
        Initialize MaskedImageDataset with preloaded data.
        
        Args:
            image_paths (list): List of image file paths.
            mask_paths (list): List of mask file paths.
            labels (list): List of labels corresponding to images.
            class_names (list): List of class names.
            transform (callable, optional): Transformations to apply to images.
        """
        self.image_paths = image_paths or []
        self.mask_paths = mask_paths or []
        self.labels = labels or []
        self.class_names = class_names or []
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]
        label = self.labels[idx]
    
        # Load images and masks
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # grayscale
    
        # Resize both image and mask
        fixed_size = (256, 256)
        image = image.resize(fixed_size, Image.BILINEAR)
        mask = mask.resize(fixed_size, Image.NEAREST)
    
        # Apply transforms
        if self.transform:
            image = self.transform(image)  # apply full transform to image
        mask = transforms.ToTensor()(mask)  # only convert mask to tensor
    
        # Convert mask to binary (0 or 1)
        mask = (mask > 0.5).float()
    
        # Ensure mask has 3 channels like the image
        mask = mask.expand(3, -1, -1)  # shape [3, 256, 256]
    
        # Apply mask to image
        masked_image = image * mask
    
        return masked_image, label
        
class ImageDataset(Dataset):
    def __init__(self, data_dir, categories, transform=None, sample_size=None):
        self.image_paths = []
        self.mask_paths = []
        self.labels = []
        self.class_names = []
        self.categories=categories
        self.transform = transform
        self.label_map = {cat: i for i, cat in enumerate(categories)}
        self.inv_label_map = {i: cat for cat, i in self.label_map.items()}  # Szám → név átalakítás

        for category in categories:
            images_path = os.path.join(data_dir, category, "images")
            masks_path = os.path.join(data_dir, category, "masks")
            
            if not os.path.isdir(images_path) or not os.path.isdir(masks_path):
                continue
            
            files = os.listdir(images_path)
            if sample_size:  # If sample_size is given, limit the number of files
                files = files[:sample_size]  
                
            for file in tqdm(files, desc=f"Loading {category} images"):
                img_path = os.path.join(images_path, file)
                mask_path = os.path.join(masks_path, file)
                
                if os.path.exists(mask_path):
                    self.image_paths.append(img_path)
                    self.mask_paths.append(mask_path)
                    self.labels.append(self.label_map[category])
                    self.class_names.append(category)

    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]
        label = self.labels[idx]
    
        # Load images and masks
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # grayscale
    
        # Resize both image and mask
        fixed_size = (256, 256)
        image = image.resize(fixed_size, Image.BILINEAR)
        mask = mask.resize(fixed_size, Image.NEAREST)
    
        # # Apply transforms
        if self.transform:
            image = self.transform(image)# apply full transform to image
        else:
            image = transforms.ToTensor()(image)
        mask = transforms.ToTensor()(mask)  # only convert mask to tensor
    
        # Convert mask to binary (0 or 1)
        mask = (mask > 0.5).float()
    
        # Ensure mask has 3 channels like the image
        mask = mask.expand(3, -1, -1)  # shape [3, 256, 256]
    
        # Ensure shape match
        assert image.shape == mask.shape, f"Shape mismatch: {image.shape} vs {mask.shape}"
    
        # Apply mask to image
        masked_image = image * mask
    
        return image,mask, label

def convert_to_masked_dataset(image_dataset, transform=None,categories=[["COVID", "Normal", "Viral Pneumonia", "Lung_Opacity"]]):
    """
    Converts an ImageDataset or its Subset object to a MaskedImageDataset object.

    Args:
        image_dataset (Dataset or Subset): The original dataset or a Subset object.
        transform (callable, optional): Transformations to apply to images in the masked dataset.

    Returns:
        MaskedImageDataset: A new dataset with masks applied.
    """
    # Handle Subset objects
    if isinstance(image_dataset, Subset):
        original_dataset = image_dataset.dataset  # Access the original dataset
        indices = image_dataset.indices  # Get subset indices
    else:
        original_dataset = image_dataset
        indices = list(range(len(original_dataset)))  # Use all indices if not a subset

    # Create a new MaskedImageDataset instance with preloaded data
    masked_dataset = MaskedImageDataset(
        image_paths=[original_dataset.image_paths[i] for i in indices],
        mask_paths=[original_dataset.mask_paths[i] for i in indices],
        labels=[original_dataset.labels[i] for i in indices],
        class_names=original_dataset.class_names,
        transform=transform
    )
    
    return masked_dataset





### A MaskedImageDataset osztály használata

In [4]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),  # ResNet bemeneti mérete
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

original_dataset = ImageDataset(data_dir='/kaggle/input/covid19-radiography-database/COVID-19_Radiography_Dataset',
                             categories=["COVID", "Normal", "Viral Pneumonia", "Lung_Opacity"],
                             transform=None,
                             sample_size=None)


num_classes = len(original_dataset.label_map)
print(num_classes)

Loading Lung_Opacity images: 100%|██████████| 6012/6012 [00:29<00:00, 201.97it/s]

4


### Adatok felosztása tanító és tesztelő halmazokra

In [5]:
# Define split ratios
train_ratio = 0.80
test_ratio = 0.10
val_ratio = 0.10

total_size = len(original_dataset)

train_size = int(train_ratio * total_size)
test_size = int(test_ratio * total_size)
val_size = int(total_size-train_size-test_size)

train_dataset_og, test_dataset_og,val_dataset_og = random_split(original_dataset, [train_size, test_size,val_size])

train_dataset=convert_to_masked_dataset(train_dataset_og, transform=transform)
test_dataset=convert_to_masked_dataset(test_dataset_og, transform=transform)
val_dataset=convert_to_masked_dataset(val_dataset_og, transform=transform)





# def count_per_category(dataset):
#     category_counts = {cat: 0 for cat in dataset.label_map.keys()}
    
#     # Wrap the loop with tqdm to show progress
#     for _, label in tqdm(dataset, desc="Counting categories"):
#         category_name = dataset.inv_label_map[label]
#         category_counts[category_name] += 1
    
#     return category_counts

# # Example usage
# print("Training Set Distribution:", count_per_category(train_dataset))
# print("Test Set Distribution:", count_per_category(test_dataset))

In [6]:
import os
from torchvision.transforms import ToPILImage
import os
from torchvision.transforms import ToPILImage

def save_dataset_to_folder(dataset, output_dir, apply_mask=False):
    """
    Saves a dataset (or Subset) to a specified folder.

    Args:
        dataset (Dataset or Subset): The dataset to save.
        output_dir (str): Path to the folder where the dataset will be saved.
        apply_mask (bool): Whether to save masked images or original images.
    """
    os.makedirs(output_dir, exist_ok=True)  # Create output directory if it doesn't exist

    # Check if the dataset is a Subset and access the original dataset if needed
    if isinstance(dataset, Subset):
        original_dataset = dataset.dataset
        indices = dataset.indices
    else:
        original_dataset = dataset
        indices = range(len(dataset))

    # Initialize transform to convert tensors to PIL images
    to_pil = ToPILImage()

    for idx in indices:
        # Extract data from the original dataset
        if apply_mask:
            image, label = original_dataset[idx]
            class_name = original_dataset.inv_label_map[label]
        else:
            image, mask, label = original_dataset[idx]  # For ImageDataset
            
            class_name = original_dataset.inv_label_map[label]

            mask_pil = to_pil(mask)
            class_dir_mask = os.path.join(output_dir, class_name,"masks")
            os.makedirs(class_dir_mask, exist_ok=True)
            # Save image
            mask_path = os.path.join(class_dir_mask, f"{idx}.png")
            mask_pil.save(mask_path)
    
        # Convert tensor to PIL image
        image_pil = to_pil(image)
    
        # Create subdirectory for each class
        class_dir_img = os.path.join(output_dir, class_name,"images")

        os.makedirs(class_dir_img, exist_ok=True)

        # Save image
        image_path = os.path.join(class_dir_img, f"{idx}.png")
        image_pil.save(image_path)

    print(f"Dataset saved successfully to {output_dir}!")


save_dataset_to_folder(test_dataset_og,"results/test_data")


Dataset saved successfully to results/test_data!


## Konvolucios háló definiálása

A ResNet-50 (Residual Network 50) egy mély konvolúciós neurális hálózat, amelyet 2015-ben mutattak be a Microsoft Research Asia kutatói. Ez az architektúra a residual block koncepció köré épül, amely lehetővé teszi rendkívül mély hálózatok hatékony tanítását és használatát.

In [7]:
from torchvision.models import googlenet
import timm

# Load pre-trained Inception-ResNet-v2 model
inception_resnet_model = timm.create_model('inception_resnet_v2', pretrained=True)

# for param in model.parameters():
#     param.requires_grad = False  # Fagyasztjuk az alapmodelt

# Utolsó teljesen kapcsolt réteg (fc) cseréje saját osztályszámra
inception_resnet_model.classif = nn.Sequential(
    nn.Linear(inception_resnet_model.classif.in_features, 512),  # Első rejtett réteg
    nn.LeakyReLU(0.01),
    nn.Dropout(0.3),
    nn.Linear(512, 256),  # Második rejtett réteg
    nn.LeakyReLU(0.01),
    nn.Dropout(0.3),
    nn.Linear(256, num_classes)  # Kimeneti réteg
)

googlenet_model = googlenet(weights='IMAGENET1K_V1')  # Load pretrained weights
googlenet_model.fc = nn.Sequential(
    nn.Linear(googlenet_model.fc.in_features, 512),  # Első rejtett réteg
    nn.LeakyReLU(0.01),
    nn.Dropout(0.3),
    nn.Linear(512, 256),  # Második rejtett réteg
    nn.LeakyReLU(0.01),
    nn.Dropout(0.3),
    nn.Linear(256, num_classes)  # Kimeneti réteg
)
resnet50_model = models.resnet50(pretrained=True)
resnet50_model.fc = nn.Sequential(
    nn.Linear(resnet50_model.fc.in_features, 512),  # Első rejtett réteg
    nn.LeakyReLU(0.01),
    nn.Dropout(0.3),
    nn.Linear(512, 256),  # Második rejtett réteg
    nn.LeakyReLU(0.01),
    nn.Dropout(0.3),
    nn.Linear(256, num_classes)  # Kimeneti réteg
)

# print(model)

model.safetensors:   0%|          | 0.00/224M [00:00<?, ?B/s]

Downloading: "https://download.pytorch.org/models/googlenet-1378be20.pth" to /root/.cache/torch/hub/checkpoints/googlenet-1378be20.pth
100%|██████████| 49.7M/49.7M [00:00<00:00, 224MB/s]
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 210MB/s]


### Logger

In [8]:
import traceback
from datetime import datetime

def log_and_upload_exception(exception, log_filename="error_log.txt", drive_folder_id=None):
    """
    Logs an exception to a file and uploads the log file to Google Drive.

    Args:
        exception (Exception): The exception object to be logged.
        log_filename (str): The name of the log file.
        drive_folder_id (str): The ID of the Google Drive folder where the log will be uploaded.
    """
    # Generate a timestamp for the log entry
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Create or append to the log file
    log_filepath = os.path.join("/kaggle/working", log_filename)
    with open(log_filepath, "a") as log_file:
        log_file.write(f"[{timestamp}] Exception Occurred:\n")
        log_file.write("".join(traceback.format_exception(type(exception), exception, exception.__traceback__)))
        log_file.write("\n" + "-" * 80 + "\n")

    print(f"Exception logged to {log_filepath}")

    # Upload the log file to Google Drive
    if drive_folder_id:
        try:
            file_metadata = {
                'name': log_filename,
                'parents': [drive_folder_id]  # Specify the target folder on Google Drive
            }
            media = MediaFileUpload(log_filepath, mimetype='text/plain')

            uploaded_file = drive_service.files().create(
                body=file_metadata,
                media_body=media,
                fields='id'
            ).execute()

            print(f"Log file uploaded to Google Drive (File ID: {uploaded_file.get('id')})")
        except Exception as upload_exception:
            print(f"Failed to upload the log file: {upload_exception}")


## K-fold tanítási algoritmus metrikák számításával

In [9]:
from collections import Counter


def train_once(model, dataset, val_dataset, test_dataset, num_classes=4, num_epochs=10, batch_size=32, lr=0.001,
               early_stop_threshold=5, lr_reduce_factor=0.1, lr_patience=3,model_name="model.pth"):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    criterion = nn.CrossEntropyLoss()

    # Split dataset into training, validation, and test sets
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Print label distribution
    print(f"Label distribution:")
    train_labels = Counter([dataset[i][1] for i in range(len(dataset))])
    val_labels = Counter([val_dataset[i][1] for i in range(len(val_dataset))])
    test_labels = Counter([test_dataset[i][1] for i in range(len(test_dataset))])
    print('Train:', train_labels)
    print('Val:', val_labels)
    print('Test:', test_labels)

    # Initialize model, optimizer, and learning rate scheduler
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=lr_reduce_factor, patience=lr_patience)

    best_model_wts = None
    best_acc = 0.0
    early_stop_count = 0

    history = {
        'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []
    }

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for inputs_batch, labels_batch in train_loader:
            inputs_batch, labels_batch = inputs_batch.to(device), labels_batch.to(device)
            optimizer.zero_grad()
            outputs_batch = model(inputs_batch)
            loss = criterion(outputs_batch, labels_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs_batch.size(0)
            _, predicted_batch = torch.max(outputs_batch, 1)
            correct += (predicted_batch == labels_batch).sum().item()
            total += labels_batch.size(0)

        train_loss_epoch = running_loss / total
        train_acc_epoch = correct / total

        # Validation phase
        model.eval()
        val_running_loss, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for inputs_val, labels_val in val_loader:
                inputs_val, labels_val = inputs_val.to(device), labels_val.to(device)
                outputs_val = model(inputs_val)
                loss_val_batch = criterion(outputs_val, labels_val)

                val_running_loss += loss_val_batch.item() * inputs_val.size(0)
                _, predicted_val = torch.max(outputs_val, 1)

                val_correct += (predicted_val == labels_val).sum().item()
                val_total += labels_val.size(0)

        val_loss_epoch = val_running_loss / val_total
        val_acc_epoch = val_correct / val_total

        print(f'Epoch {epoch+1}, Train Loss: {train_loss_epoch:.4f}, Train Acc: {train_acc_epoch:.4f}, '
              f'Val Loss: {val_loss_epoch:.4f}, Val Acc: {val_acc_epoch:.4f}')

        # Store metrics for this epoch
        history['train_loss'].append(train_loss_epoch)
        history['train_acc'].append(train_acc_epoch)
        history['val_loss'].append(val_loss_epoch)
        history['val_acc'].append(val_acc_epoch)

        # Check if this is the best model so far
        if val_acc_epoch > best_acc:
            best_acc = val_acc_epoch
            best_model_wts = copy.deepcopy(model.state_dict())
            early_stop_count = 0
        else:
            early_stop_count += 1

        if early_stop_count >= early_stop_threshold:
            print(f"Early stopping at epoch {epoch+1}")
            break

        scheduler.step(val_loss_epoch)

    # Load best model weights and evaluate on the test set
    model.load_state_dict(best_model_wts)
    
    print("\nEvaluating on test set...")
    model.eval()
    test_correct, test_total = 0, 0
    y_true_test, y_pred_test = [], []

    with torch.no_grad():
        for inputs_test, labels_test in test_loader:
            inputs_test, labels_test = inputs_test.to(device), labels_test.to(device)
            outputs_test = model(inputs_test)
            _, predicted_test = torch.max(outputs_test, 1)

            test_correct += (predicted_test == labels_test).sum().item()
            test_total += labels_test.size(0)

            y_true_test.extend(labels_test.cpu().numpy())
            y_pred_test.extend(predicted_test.cpu().numpy())

    test_acc = test_correct / test_total
    class_report_test = classification_report(
        y_true_test,
        y_pred_test,
        labels=list(range(num_classes)),
        output_dict=True,
        zero_division=0,
    )
    
    conf_matrix_test = confusion_matrix(y_true_test, y_pred_test)

    print(f"\nTest Accuracy: {test_acc:.4f}")
    print("Classification Report:")
    print(classification_report(y_true_test, y_pred_test))
    
    print("Confusion Matrix:")
    print(conf_matrix_test)
    torch.save(model.state_dict(), model_name)


    return {
        'test_accuracy': test_acc,
        'classification_report': class_report_test,
        'confusion_matrix': conf_matrix_test.tolist(),
        'history': history,
    }


### K-fold alkalmazása

In [10]:
folder_id = "1LVJ2nsLuiiOB_4qJP5xbgRgYl-cs6MTU"
metrics_results = []
histories_results = []
fold_results = {}  # Új változó a harmadik visszatérési értékhez

try:
    # Assuming dataset is already split into train/val/test subsets:
    results_inc_resnet_v2 = train_once(
        model=inception_resnet_model,
        dataset=train_dataset,
        val_dataset=val_dataset,
        test_dataset=test_dataset,
        num_classes=4,
        num_epochs=20,
        model_name="results/inc_resnet_v2.pth"

    )
    results_google_net = train_once(
        model=googlenet_model,
        dataset=train_dataset,
        val_dataset=val_dataset,
        test_dataset=test_dataset,
        num_classes=4,
        num_epochs=20,
        model_name="results/google_net.pth"

    )   
    results_resnet50 = train_once(
        model=resnet50_model,
        dataset=train_dataset,
        val_dataset=val_dataset,
        test_dataset=test_dataset,
        num_classes=4,
        num_epochs=20,
        model_name="results/resnet50.pth"
    )

    print("Final Test Accuracy:", results['test_accuracy'])
    print("Classification Report:", results['classification_report'])

except Exception as e:
        log_and_upload_exception(e, drive_folder_id=folder_id)
        raise(e)

Label distribution:
Train: Counter({1: 8136, 3: 4798, 0: 2902, 2: 1096})
Val: Counter({1: 1026, 3: 610, 0: 356, 2: 125})
Test: Counter({1: 1030, 3: 604, 0: 358, 2: 124})
Epoch 1, Train Loss: 0.5635, Train Acc: 0.7912, Val Loss: 0.4219, Val Acc: 0.8545
Epoch 2, Train Loss: 0.3686, Train Acc: 0.8730, Val Loss: 0.3471, Val Acc: 0.8781
Epoch 3, Train Loss: 0.2948, Train Acc: 0.8988, Val Loss: 0.4116, Val Acc: 0.8526
Epoch 4, Train Loss: 0.2527, Train Acc: 0.9143, Val Loss: 0.7291, Val Acc: 0.8427
Epoch 5, Train Loss: 0.2124, Train Acc: 0.9269, Val Loss: 0.3421, Val Acc: 0.8725
Epoch 6, Train Loss: 0.1956, Train Acc: 0.9331, Val Loss: 0.6429, Val Acc: 0.8120
Epoch 7, Train Loss: 0.1693, Train Acc: 0.9419, Val Loss: 0.3740, Val Acc: 0.8739
Early stopping at epoch 7

Evaluating on test set...

Test Accuracy: 0.8918
Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.70      0.81       358
           1       0.87      0.97      0.92     

NameError: name 'results' is not defined

## Legjobb model betoltese es kiertekelese - kezdetleges, jelenleg lehet felesleges

In [ ]:

# # Funkció a modell betöltésére
# def load_model(model_path):
#     model = torch.load(model_path)  # Modell betöltése (architektúra + súlyok)
#     model.eval()  # Eval módba állítás
#     print("Model loaded successfully.")
#     return model

# # Funkció a modell kiértékelésére és az eredmények mentésére Excelbe
# def evaluate_and_save_results(model, test_dataset, results_path="evaluation_results.xlsx", batch_size=32):
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     model.to(device)
    
#     # Teszt adathalmaz betöltése
#     test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
#     y_true = []
#     y_pred = []
    
#     with torch.no_grad():
#         for inputs, labels in test_loader:
#             inputs, labels = inputs.to(device), labels.to(device)
            
#             # Előrejelzés
#             outputs = model(inputs)
#             _, predicted = torch.max(outputs, 1)
            
#             y_true.extend(labels.cpu().numpy())
#             y_pred.extend(predicted.cpu().numpy())
    
#     # Metrikák kiszámítása
#     accuracy = accuracy_score(y_true, y_pred)
#     class_report = classification_report(y_true, y_pred, output_dict=True)
#     conf_matrix = confusion_matrix(y_true, y_pred)

#     # Kiértékelési eredmények mentése Excel fájlba
#     with pd.ExcelWriter(results_path, engine='openpyxl') as writer:
#         # Pontosság mentése
#         accuracy_df = pd.DataFrame({"Metric": ["Accuracy"], "Value": [accuracy]})
#         accuracy_df.to_excel(writer, sheet_name="Accuracy", index=False)

#         # Osztályonkénti metrikák mentése
#         metrics_data = []
#         for cls in class_report:
#             if cls.isdigit():  # Csak az osztályokra vonatkozó metrikák mentése
#                 metrics_data.append({
#                     "Class": cls,
#                     "Precision": class_report[cls]["precision"],
#                     "Recall": class_report[cls]["recall"],
#                     "F1-Score": class_report[cls]["f1-score"]
#                 })
#         metrics_df = pd.DataFrame(metrics_data)
#         metrics_df.to_excel(writer, sheet_name="Class Metrics", index=False)

#         # Konfúziós mátrix mentése
#         conf_matrix_df = pd.DataFrame(conf_matrix)
#         conf_matrix_df.to_excel(writer, sheet_name="Confusion Matrix", index=True)

#     print(f"Evaluation results saved to {results_path}")

# # Példa használat
# model_path = 'results/best_model.pth'  # A legjobb modell elérési útja
# test_dataset = ...  # Teszt adathalmaz (adathalmazt itt kell definiálni)

# # Modell betöltése és kiértékelése
# model = load_model(model_path)
# evaluate_and_save_results(model, test_dataset, results_path="evaluation_results.xlsx")

## Adatok mentése Google-Drive-ra

### Mentés mint excel állomány

In [11]:
def export_results_to_excel(results, filename='model_results.xlsx'):
    import pandas as pd

    # Extract results
    test_accuracy = results['test_accuracy']
    classification_report = results['classification_report']
    confusion_matrix = results['confusion_matrix']
    history = results['history']

    # Create Excel writer
    with pd.ExcelWriter(filename) as writer:
        # 1. Summary sheet with overall metrics
        summary_data = {
            'Metric': ['Test Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1', 
                       'Weighted Precision', 'Weighted Recall', 'Weighted F1'],
            'Value': [
                test_accuracy,
                classification_report['macro avg']['precision'],
                classification_report['macro avg']['recall'],
                classification_report['macro avg']['f1-score'],
                classification_report['weighted avg']['precision'],
                classification_report['weighted avg']['recall'],
                classification_report['weighted avg']['f1-score']
            ]
        }

        # Add per-class metrics
        class_names = list(classification_report.keys())[:-3]  # Exclude 'accuracy', 'macro avg', and 'weighted avg'
        for class_name in class_names:
            summary_data['Metric'].extend([
                f'{class_name} Precision',
                f'{class_name} Recall',
                f'{class_name} F1'
            ])
            summary_data['Value'].extend([
                classification_report[class_name]['precision'],
                classification_report[class_name]['recall'],
                classification_report[class_name]['f1-score']
            ])

        summary_df = pd.DataFrame(summary_data)
        summary_df.to_excel(writer, sheet_name='Summary', index=False)

        # 2. Confusion matrix sheet
        conf_df = pd.DataFrame(
            confusion_matrix,
            index=[f'True {c}' for c in class_names],
            columns=[f'Pred {c}' for c in class_names]
        )
        conf_df.to_excel(writer, sheet_name='Confusion Matrix')

        # 3. Training history sheet
        history_data = {
            'Epoch': list(range(1, len(history['train_loss']) + 1)),
            'Train Loss': history['train_loss'],
            'Train Acc': history['train_acc'],
            'Val Loss': history['val_loss'],
            'Val Acc': history['val_acc']
        }
        history_df = pd.DataFrame(history_data)
        history_df.to_excel(writer, sheet_name='Training History', index=False)

    print(f"Results successfully exported to {filename}")

export_results_to_excel(results_inc_resnet_v2, filename='results/results_inc_resnet_v2.xlsx')
export_results_to_excel(results_google_net, filename='results/results_google_net.xlsx')
export_results_to_excel(results_resnet50, filename='results/results_resnet50.xlsx')


Results successfully exported to results/results_inc_resnet_v2.xlsx
Results successfully exported to results/results_google_net.xlsx
Results successfully exported to results/results_resnet50.xlsx


In [ ]:
import os
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

# JSON key file path
SERVICE_ACCOUNT_FILE = '/kaggle/input/googledriveuploadauth/covidcxr-0aaa95b10e20.json'

# Authenticate
credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE,
    scopes=['https://www.googleapis.com/auth/drive']
)

# Create Google Drive API client
drive_service = build('drive', 'v3', credentials=credentials)

# Source directory on Kaggle (to upload)
source_dir = '/kaggle/working/results'

# Parent folder ID on Google Drive (where the new folder will be created)
parent_folder_id = '1LVJ2nsLuiiOB_4qJP5xbgRgYl-cs6MTU'  # Replace with your parent folder ID

# Target folder name on Google Drive
target_folder_name = '4application'

# Function to get or create a folder in Google Drive
def get_or_create_folder(folder_name, parent_id):
    query = f"name='{folder_name}' and mimeType='application/vnd.google-apps.folder' and '{parent_id}' in parents"
    results = drive_service.files().list(q=query, fields="files(id, name)").execute()
    items = results.get('files', [])
    
    if items:
        # If the folder exists, return its ID
        return items[0]['id']
    else:
        # If the folder doesn't exist, create it
        file_metadata = {
            'name': folder_name,
            'mimeType': 'application/vnd.google-apps.folder',
            'parents': [parent_id]
        }
        folder = drive_service.files().create(body=file_metadata, fields='id').execute()
        return folder.get('id')

# Recursive function to upload files and folders
def upload_folder_to_drive(local_folder_path, drive_parent_id):
    for item in os.listdir(local_folder_path):
        item_path = os.path.join(local_folder_path, item)
        
        if os.path.isdir(item_path):
            # If it's a directory, create a corresponding folder in Google Drive
            folder_id = get_or_create_folder(item, drive_parent_id)
            # Recursively upload the contents of the directory
            upload_folder_to_drive(item_path, folder_id)
        elif os.path.isfile(item_path):
            # If it's a file, upload it to Google Drive
            file_metadata = {
                'name': item,
                'parents': [drive_parent_id]
            }
            media = MediaFileUpload(item_path, mimetype='application/octet-stream')
            
            try:
                file = drive_service.files().create(
                    body=file_metadata,
                    media_body=media,
                    fields='id'
                ).execute()
                print(f"Uploaded: {item} (File ID: {file.get('id')})")
            except Exception as e:
                print(f"Error uploading {item}: {e}")

# Check if source directory exists
if not os.path.exists(source_dir):
    print(f"Error: The source directory '{source_dir}' does not exist.")
else:
    # Get or create the target folder in Google Drive
    target_folder_id = get_or_create_folder(target_folder_name, parent_folder_id)
    
    # Upload the entire directory structure to Google Drive
    upload_folder_to_drive(source_dir, target_folder_id)


Uploaded: inc_resnet_v2.pth (File ID: 13ZEVlxKw72T4xKIjV-C-gJHWue5de_B5)
Uploaded: results_resnet50.xlsx (File ID: 19zFMdMB8TxM2nkw4ryixK-GgsHS3oAPe)
Uploaded: results_google_net.xlsx (File ID: 1WyswahvWJ8dxsnPpdokh2FTCPIczyDOu)
Uploaded: google_net.pth (File ID: 1mIbY0LETYDg2TtuIVitpYDUc2_1KRmhH)
Uploaded: results_inc_resnet_v2.xlsx (File ID: 1g3ZklIbiVp9tIf3G26SjZiNxXgw-ddAq)
Uploaded: 5965.png (File ID: 1vrt01Z0vrQYEv8aINrGDhDMNHF2PVoHN)
Uploaded: 4042.png (File ID: 1hil7QqGEkIHWiZ5w_rnTw3TbNmSxNmIi)
Uploaded: 3909.png (File ID: 18Q_h7Sfi5tFf6mpIVP600fLkO7lyRHcD)
Uploaded: 5387.png (File ID: 16uOW0wysh64WRnLpeEUziobAyeaNopjx)
Uploaded: 12560.png (File ID: 1sENyDTidY765nHLJ1wn09X2TsA4WEzLG)
Uploaded: 11693.png (File ID: 19oRBaGdxpEfYIXS8N9OIHOPOeFyk148I)
Uploaded: 12396.png (File ID: 1gnIAcogRQz7sWyH1KwfFWVmsHM9CDxbf)
Uploaded: 8261.png (File ID: 1ugZVKVdYwPneJlmsFSRBTYyk-T9d978F)
Uploaded: 3680.png (File ID: 1EjPGv11cWt51ML7wTmtkLVvxpvmZib4k)
Uploaded: 4692.png (File ID: 1AEZbUwtLbB

# Letoltes drive-rol

In [ ]:
# import io
# import os
# from google.oauth2 import service_account
# from googleapiclient.discovery import build
# from googleapiclient.http import MediaIoBaseDownload

# # JSON kulcs fájl elérési útja
# SERVICE_ACCOUNT_FILE = '/kaggle/input/googledriveuploadauth/covidcxr-0aaa95b10e20.json'

# # Hitelesítés
# credentials = service_account.Credentials.from_service_account_file(
#     SERVICE_ACCOUNT_FILE,
#     scopes=['https://www.googleapis.com/auth/drive.readonly']
# )

# # Google Drive API kliens létrehozása
# drive_service = build('drive', 'v3', credentials=credentials)

# # Forrásmappa ID a Google Drive-on
# source_folder_id = '1LVJ2nsLuiiOB_4qJP5xbgRgYl-cs6MTU'  # Cseréld ki a megfelelő mappa ID-ra

# # Célmappa a letöltött fájloknak
# download_dir = '/kaggle/working/downloaded_files'
# os.makedirs(download_dir, exist_ok=True)

# # Rekurzív fájl letöltés mappából
# def download_folder_contents(folder_id, local_path):
#     # Fájlok és mappák lekérése a megadott mappából
#     query = f"'{folder_id}' in parents and trashed = false"
#     results = drive_service.files().list(
#         q=query,
#         fields="nextPageToken, files(id, name, mimeType)"
#     ).execute()
    
#     items = results.get('files', [])
    
#     if not items:
#         print(f"A mappa üres vagy nem található.")
#         return
    
#     # Fájlok és mappák feldolgozása
#     for item in items:
#         item_id = item['id']
#         item_name = item['name']
#         item_mime_type = item['mimeType']
        
#         # Elérési út létrehozása
#         item_path = os.path.join(local_path, item_name)
        
#         # Ha mappa, akkor rekurzívan letöltjük a tartalmát
#         if item_mime_type == 'application/vnd.google-apps.folder':
#             print(f"Mappa feldolgozása: {item_name}")
#             os.makedirs(item_path, exist_ok=True)
#             download_folder_contents(item_id, item_path)
#         else:
#             # Ha fájl, akkor letöltjük
#             try:
#                 request = drive_service.files().get_media(fileId=item_id)
                
#                 with open(item_path, 'wb') as f:
#                     downloader = MediaIoBaseDownload(f, request)
#                     done = False
#                     while not done:
#                         status, done = downloader.next_chunk()
#                         print(f"Letöltés: {item_name} - {int(status.progress() * 100)}%")
                
#                 print(f"Letöltve: {item_name}")
#             except Exception as e:
#                 print(f"Hiba történt a '{item_name}' letöltése során: {e}")

# # Mappa tartalmának letöltése
# print(f"A '{source_folder_id}' mappa tartalmának letöltése ide: {download_dir}")
# download_folder_contents(source_folder_id, download_dir)
# print(f"A letöltés befejeződött. A fájlok itt találhatók: {download_dir}")


